# Capstone — Forecasting Stock Prices in Malaysia
## Financial News Sentiment Data — Web Scraping Pipeline

This notebook collects financial news articles from **The Star** for the top 10 stocks listed on Bursa Malaysia.
Collected articles are stored in a local **SQLite** database for use in the downstream sentiment analysis pipeline.

### How this notebook is structured
| Section | What it does |
|---|---|
| 1. Imports & Configuration | Load libraries and define all settings in one place |
| 2. Database Setup | Create the database table that stores articles |
| 3. The Star — Metadata Scraper | Collect article titles, links, and dates via the Queryly API |
| 4. The Star — Body Text Scraper | Visit each article page and extract the full article text |
| 5. Verification & Summary | Inspect what was collected |

---
## 1. Imports & Configuration

### 1.1 — Library Imports

Before we can run any code, we need to load the external tools (called **libraries**) that our script depends on.
Think of libraries as toolboxes — each one gives us a set of ready-made functions so we don't have to build everything from scratch.

| Library | What it does |
|---|---|
| `time` | Lets us pause the script between requests using `time.sleep()` — this prevents us from sending too many requests too quickly and getting blocked |
| `requests` | Sends HTTP requests to websites and APIs — this is how we actually "visit" a URL in code |
| `sqlite3` | Lets us create and interact with a local SQLite database file to store collected articles |
| `datetime` | Lets us work with dates and times — used to convert article publish dates into Unix timestamps (a universal numeric date format) |
| `BeautifulSoup` (from `bs4`) | Parses raw HTML from a webpage and lets us search for specific elements like `<div>` or `<p>` tags |
| `urllib3` | A lower-level library used here only to suppress SSL certificate warnings |

In [1]:
import time
import requests
import sqlite3
import datetime
from bs4 import BeautifulSoup
import urllib3

# Suppress SSL warnings. Safe here since we are only reading public news pages.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

### 1.2 — Configuration

All settings for the scraper are defined here in one place.
Keeping configuration at the top means that if you need to change something — such as adding a new stock or adjusting the cutoff date — you only need to edit this one cell rather than hunting through the whole notebook.

**Key settings explained:**
- `DB_PATH` — the file path where the SQLite database will be saved. Change this if you want the database stored in a different folder.
- `API_KEY` — the key needed to use The Star's internal Queryly search API. This was identified by inspecting network traffic on thestar.com.my.
- `CUTOFF_DATE` — any article older than this date will be ignored. `datetime.datetime(...).timestamp()` converts a human-readable date into a Unix timestamp (the number of seconds since 1 January 1970) so it can be compared directly with the `pubdateunix` values returned by the API.
- `STOCKS` — a dictionary mapping each stock's Bursa ticker code to a plain-English search term. `STOCKS.items()` is used later to loop through all tickers and their matching search queries together.
- `BATCH_SIZE` — how many articles to request per API call. The maximum tested is 100.
- `THESTAR_BODY` — stores the HTML attribute and value used to locate the article body `<div>` on The Star's article pages. Storing this as a dictionary makes it easy to pass into `fetch_thestar_body()` without hardcoding the values inside the function.

In [2]:
# ── File Paths ─────────────────────────────────────────────────────────────────
DB_PATH = '../data/database.db'

# ── The Star API Settings ──────────────────────────────────────────────────────
# The Star uses the Queryly search API internally. These URLs and the API key
# were identified by inspecting the network requests made by thestar.com.my.
API_BASE_URL        = 'https://api.queryly.com/json.aspx'
API_KEY             = '6ddd278bf17648ac'
EXTENDED_DATAFIELDS = 'paywalltype,kicker'
BATCH_SIZE          = 100  # Articles per API call. Maximum tested is 100.

# ── Date Cutoff ────────────────────────────────────────────────────────────────
# Articles published before this date will be ignored.
# datetime.datetime(...).timestamp() converts a human-readable date into a
# Unix timestamp (seconds since 1 Jan 1970) so it can be compared directly
# with the pubdateunix values returned by the API.
CUTOFF_DATE = datetime.datetime(2016, 1, 1).timestamp()

# ── Stock Tickers ──────────────────────────────────────────────────────────────
# Maps each Bursa Malaysia ticker to a plain-English search term.
# The search term is passed to the API to find relevant articles.
# Format: 'TICKER': 'search term'
STOCKS = {
    '1155.KL': 'maybank',
    '1023.KL': 'cimb',
    '1295.KL': 'public bank',
    '5211.KL': 'sunway',
    '5225.KL': 'ihh healthcare',
    '5285.KL': 'sd guthrie',
    '5347.KL': 'tenaga nasional',
    '5819.KL': 'hong leong bank',
    '6947.KL': 'celcomdigi',
    '8869.KL': 'press metal',
}

# ── HTML Body Locator ──────────────────────────────────────────────────────────
# The Star wraps its article content in a specific <div> identified by its id.
# This dictionary stores the attribute name and value needed to find that div.
# It is passed into fetch_thestar_body() to keep the values in one place.
THESTAR_BODY = {'attribute': 'id', 'value': 'story-body'}

---
## 2. Database Setup

### 2.1 — Initialise the Database

This function creates the SQLite database file and sets up the table that will store all collected articles.
You only need to run this once — but it is safe to run again because of the `IF NOT EXISTS` guard.

**How it works, step by step:**
1. `sqlite3.connect(DB_PATH)` — opens a connection to the database file. If the file doesn't exist yet, SQLite creates it automatically.
2. `connection.cursor()` — creates a **cursor**, which is the object we use to send SQL commands to the database.
3. `cursor.execute(''' CREATE TABLE IF NOT EXISTS ... ''')` — sends a SQL command that creates the `article_table` table. `IF NOT EXISTS` means this command is silently ignored if the table is already there, preventing errors on repeated runs.
4. `connection.commit()` — saves all changes made during this connection to the database file on disk. Without this, changes would be lost when the connection closes.
5. `connection.close()` — closes the connection cleanly to prevent file corruption or data locks.

**Table columns explained:**

| Column | Type | Description |
|---|---|---|
| `ticker` | TEXT | The Bursa Malaysia stock ticker (e.g. `1155.KL`) |
| `article_id` | INTEGER | Unique ID assigned by The Star's API |
| `source` | TEXT | Which news source the article came from (currently always `thestar`) |
| `title` | TEXT | Headline of the article |
| `link` | TEXT | Full URL to the article page |
| `content` | TEXT | The article's full body text (filled in by the body scraper) |
| `pubdateunix` | INTEGER | Publication date as a Unix timestamp |
| `kicker` | TEXT | Section label (e.g. `Banking`, `Business`) |
| `paywalltype` | TEXT | Whether the article is free (`Complimentary`) or paywalled |
| `body_fetched` | INTEGER | Flag: `0` = body text not yet collected, `1` = body text collected |

**The primary key uses three columns:** `(ticker, article_id, source)`.
Including `source` future-proofs the table in case additional news sources are added later —
it ensures records from different sources can never collide even if they share the same numeric article ID.

In [3]:
def initialise_database():
    connection = sqlite3.connect(DB_PATH)
    cursor = connection.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS article_table (
            ticker       TEXT,
            article_id   INTEGER,
            source       TEXT,
            title        TEXT,
            link         TEXT,
            content      TEXT,
            pubdateunix  INTEGER,
            kicker       TEXT,
            paywalltype  TEXT,
            body_fetched INTEGER DEFAULT 0,
            PRIMARY KEY (ticker, article_id, source)
        )
    ''')
    connection.commit()
    connection.close()

### 2.2 — Verify the Database (Optional)

Run this cell after calling `initialise_database()` to confirm that the table was created successfully.
`sqlite_master` is a special built-in SQLite table that stores metadata about all tables in the database.
If you see `[('article_table',)]` in the output, everything is set up correctly.

In [4]:
def verify_database():
    """Prints all table names currently in the database. Used to confirm setup."""
    connection = sqlite3.connect(DB_PATH)
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    print(cursor.fetchall())
    connection.close()

# Uncomment the line below to run the check:
verify_database()

[('article_table',)]


---
## 3. The Star — Metadata Scraper

This section collects article **metadata** (title, link, publish date, kicker, paywall type) for each stock
using The Star's internal Queryly search API. The article body text is collected separately in Section 4.

### Why two separate steps?
The API returns metadata for thousands of articles very efficiently — one call gets up to 100 records at once.
However, fetching the actual body text requires visiting each article's page individually, which is much slower
(we add a 1-second delay between each to avoid overloading the server). Separating the two steps means
if the body-fetching step is interrupted, we don't lose the metadata we already collected.

### 3.1 — Fetch a Batch of Articles from the API

`fetch_articles(query, endindex)` sends one request to the Queryly API and returns a batch of up to `BATCH_SIZE` articles.

**Parameters:**
- `query` — the search term (e.g. `'maybank'`)
- `endindex` — the starting position in the results list. This is how we paginate: `0` gets articles 1–100, `100` gets 101–200, and so on.

**How it works:**
- `params` — a dictionary of URL query parameters. `requests.get()` automatically appends these to the URL.
- `response.raise_for_status()` — throws an error immediately if the server returns a failure status code (e.g. 404 or 500), so problems are caught early rather than silently continuing with bad data.
- `response.json()` — converts the raw JSON text response into a Python dictionary we can work with.
- The `try/except` block catches any errors (network failures, bad responses) and prints them rather than crashing the whole scraper.

In [ ]:
def fetch_articles(query, endindex):
    """
    Fetches one batch of article metadata from The Star's Queryly search API.

    Args:
        query (str):    The search term to look up (e.g. 'maybank').
        endindex (int): The result offset — used to page through results in batches.

    Returns:
        dict: The API response as a Python dictionary, or None if the request failed.
    """
    try:
        params = {
            'queryly_key':        API_KEY,
            'query':              query,
            'endindex':           endindex,
            'batchsize':          BATCH_SIZE,
            'extendeddatafields': EXTENDED_DATAFIELDS
        }
        response = requests.get(API_BASE_URL, params=params)
        response.raise_for_status()  # Raises an error if the server returned a failure code
        return response.json()       # Convert the response text from JSON into a Python dict
    except Exception as e:
        print(f"Error fetching articles for '{query}' at index {endindex}: {e}")
        return None

### 3.2 — Save a Batch of Articles to the Database

`save_thestar_articles(ticker, data)` takes the API response from `fetch_articles()` and inserts each article record into the database.

**How it works:**
- We loop through each item in `data['items']` — the list of article records returned by the API.
- Before saving, we check `article['pubdateunix'] < CUTOFF_DATE`. Since the API returns articles newest-first, the moment we hit an article older than our cutoff we can stop — all remaining articles will be even older. We signal this to the caller by returning `True` as the second value.
- `INSERT OR IGNORE` — inserts the record, but silently skips it if a record with the same primary key already exists. This makes re-running the scraper safe.
- The `?` placeholders in the SQL statement are filled in by the values tuple. This is called **parameterised queries** and prevents SQL injection errors.
- The `try/finally` block ensures `connection.commit()` and `connection.close()` are always called, even if an error occurs mid-loop.

In [ ]:
def save_thestar_articles(ticker, data):
    """
    Saves a batch of The Star article metadata to the database.

    Stops early and signals the caller if an article older than CUTOFF_DATE is found,
    since the API returns results newest-first.

    Args:
        ticker (str): The stock ticker this batch of articles is associated with.
        data (dict):  The API response dictionary containing an 'items' list.

    Returns:
        tuple: (count, reached_cutoff)
            - count (int):           Number of articles saved in this batch.
            - reached_cutoff (bool): True if we hit the cutoff date and should stop paginating.
    """
    connection = sqlite3.connect(DB_PATH)
    cursor = connection.cursor()
    count = 0
    try:
        for article in data['items']:
            # API returns articles newest-first, so once we pass the cutoff date
            # we can stop — all remaining articles will be even older.
            if article['pubdateunix'] < CUTOFF_DATE:
                return count, True  # Signal to the caller: stop paginating

            cursor.execute('''
                INSERT OR IGNORE INTO article_table
                    (ticker, article_id, source, title, link, content, pubdateunix, kicker, paywalltype)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                ticker,
                article['_id'],
                'thestar',
                article['title'],
                article['link'],
                article['description'],  # Short snippet — full body fetched later
                article['pubdateunix'],
                article['kicker'],
                article['paywalltype'],
            ))
            count += 1

        return count, False  # Cutoff not reached — caller should keep paginating

    finally:
        # Always commit and close, even if an error occurred above
        connection.commit()
        connection.close()

### 3.3 — Scrape All Articles for One Stock

`scrape_thestar_stock(ticker, query)` ties `fetch_articles()` and `save_thestar_articles()` together into a loop that pages through all available results for a single stock.

**How the pagination loop works:**
- `endindex` starts at `0` and increases by `BATCH_SIZE` (100) after each successful batch.
- The loop stops when one of three things happens:
  1. `fetch_articles()` returns `None` (a network error occurred)
  2. The API returns an empty `items` list (no more results available)
  3. `save_thestar_articles()` returns `True` for `reached_cutoff` (we've gone back far enough in time)

In [ ]:
def scrape_thestar_stock(ticker, query):
    """
    Collects all available article metadata for a single stock from The Star.
    Pages through results in batches until the cutoff date is reached or no
    more articles are available.

    Args:
        ticker (str): The Bursa Malaysia ticker code (e.g. '1155.KL').
        query (str):  The search term used to find relevant articles (e.g. 'maybank').
    """
    print(f'Starting metadata collection for: {ticker} ({query})')
    endindex       = 0
    reached_cutoff = False
    total_count    = 0

    while not reached_cutoff:
        data = fetch_articles(query, endindex)

        if data is None:
            print(f'  Network error — stopping early for {ticker}')
            break

        if len(data['items']) == 0:
            print(f'  No more results returned by API for {ticker}')
            break

        count, reached_cutoff = save_thestar_articles(ticker, data)
        endindex    += BATCH_SIZE
        total_count += count
        print(f'  Progress: {total_count} articles collected for {ticker}')

    print(f'Finished {ticker} — total articles collected: {total_count}\n')

### 3.4 — Run the Metadata Scraper

`scrape_thestar_all()` calls `scrape_thestar_stock()` for every stock in the `STOCKS` dictionary.
`STOCKS.items()` returns each key-value pair as `(ticker, query)` so we can unpack both in the loop.

> ⚠️ **This will take several minutes to run** since it collects thousands of articles.
> The database is updated after each batch, so progress is saved even if the notebook is interrupted.

In [ ]:
def scrape_thestar_all():
    """Initialises the database and runs the metadata scraper for all stocks."""
    initialise_database()
    for ticker, query in STOCKS.items():
        scrape_thestar_stock(ticker, query)
    print('Metadata collection complete for all stocks.')

# Uncomment to run:
# scrape_thestar_all()

---
## 4. The Star — Article Body Text Scraper

The metadata scraper (Section 3) only collected article headlines, links, and short descriptions.
This section visits each article's full URL and extracts the complete body text.

**Only free (non-paywalled) articles are fetched** — paywalled articles cannot be accessed without a
subscription, so they are filtered out using `WHERE paywalltype = 'Complimentary'`.

**`body_fetched`** acts as a progress flag. When a body is successfully scraped, the flag is set to `1`.
This means the scraper can be safely interrupted and restarted — it will pick up from where it left off
rather than re-scraping articles already processed.

### 4.1 — Fetch and Save Body Text

**How it works, step by step:**
1. The function first queries the database for all free articles not yet fetched, loading them all into memory with `fetchall()` before the loop starts. This is important — it keeps the cursor free to run `UPDATE` queries inside the loop without interfering with the original query.
2. For each article, it visits the article URL using `requests.get()`.
3. `BeautifulSoup(response.text, 'html.parser')` — parses the HTML of the page so we can search it like a document.
4. `soup.find('div', {div_attribute: div_value})` — searches for the `<div>` that contains the article body using the locator values from `THESTAR_BODY`.
5. `story_div.find_all('p')` — finds all paragraph tags inside the body div.
6. `' '.join([p.get_text() for p in paragraphs])` — extracts the plain text from each paragraph and joins them into a single string.
7. `time.sleep(1)` — waits 1 second between each request to be a polite scraper and avoid getting IP-blocked.
8. Progress is committed to the database every 100 articles to protect against data loss if the scraper is interrupted.

In [ ]:
def fetch_thestar_body(div_attribute, div_value):
    """
    Visits each Star article page and scrapes the full body text for all free
    articles that have not yet been fetched.

    The body text is saved back to the database and the body_fetched flag is
    set to 1 to mark the article as complete.

    Args:
        div_attribute (str): The HTML attribute used to locate the article body div
                             (e.g. 'id' for The Star).
        div_value (str):     The value of that attribute (e.g. 'story-body').
    """
    connection = sqlite3.connect(DB_PATH)
    cursor = connection.cursor()
    count = 0

    try:
        # Load all unfetched free articles into memory before the loop,
        # so the cursor remains free to run UPDATE queries inside the loop.
        cursor.execute('''
            SELECT article_id, link
            FROM article_table
            WHERE source = 'thestar'
            AND paywalltype = 'Complimentary'
            AND body_fetched = 0
        ''')
        articles_to_fetch = cursor.fetchall()
        print(f'Found {len(articles_to_fetch)} articles still needing body text')

        for article_id, link in articles_to_fetch:
            try:
                time.sleep(1)  # Be polite — wait 1 second between requests

                response = requests.get(link)
                response.raise_for_status()

                # Parse the HTML and locate the article body div
                soup = BeautifulSoup(response.text, 'html.parser')
                story_div = soup.find('div', {div_attribute: div_value})

                if story_div is None:
                    print(f'  Could not find body div for article_id {article_id}: {link}')
                    continue  # Skip this article and move on to the next one

                # Extract plain text from all <p> (paragraph) tags in the body div
                paragraphs = story_div.find_all('p')
                text = ' '.join([p.get_text() for p in paragraphs])

                # Save the body text and mark this article as fetched
                cursor.execute('''
                    UPDATE article_table
                    SET content = ?, body_fetched = 1
                    WHERE article_id = ? AND source = 'thestar'
                ''', (text, article_id))

                count += 1
                # Commit to disk every 100 articles to protect against data loss
                if count % 100 == 0:
                    print(f'  Progress: {count} article bodies fetched')
                    connection.commit()

            except Exception as e:
                print(f'  Error fetching article_id {article_id}: {e}')

    finally:
        # Always commit and close, even if an error interrupted the loop
        print(f'Total article bodies fetched: {count}')
        connection.commit()
        connection.close()

### 4.2 — Run the Body Scraper

> ⚠️ **This will take several hours** due to the 1-second delay between each of the ~27,000 articles.
> Progress is saved every 100 articles, so it is safe to stop and resume at any time.

In [ ]:
# Uncomment to run:
# fetch_thestar_body(THESTAR_BODY['attribute'], THESTAR_BODY['value'])

---
## 5. Verification & Summary

Use these cells to inspect the database contents at any point.
They do not modify any data — they only read from the database.

### 5.1 — Article Count by Ticker

In [5]:
# How many articles have been collected for each stock?
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
cursor.execute('''
    SELECT ticker, COUNT(*) AS article_count
    FROM article_table
    GROUP BY ticker
    ORDER BY ticker
''')
for row in cursor.fetchall():
    print(row)
connection.close()

('1023.KL', 6671)
('1155.KL', 7585)
('1295.KL', 13959)
('5211.KL', 5337)
('5225.KL', 1127)
('5285.KL', 334)
('5347.KL', 4456)
('5819.KL', 4618)
('6947.KL', 1731)
('8869.KL', 1309)


### 5.2 — Date Range per Ticker

In [6]:
# What is the earliest and latest article date for each stock?
# MIN/MAX are applied to the Unix timestamps, so the output is still in Unix format.
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
cursor.execute('''
    SELECT ticker,
           MIN(pubdateunix) AS oldest_article,
           MAX(pubdateunix) AS newest_article,
           COUNT(*)         AS total
    FROM article_table
    GROUP BY ticker
    ORDER BY ticker
''')
for row in cursor.fetchall():
    print(row)
connection.close()

('1023.KL', 1577833200, 1773961200, 6671)
('1155.KL', 1577833200, 1773961200, 7585)
('1295.KL', 1564873200, 1773969660, 13959)
('5211.KL', 1577833200, 1773961200, 5337)
('5225.KL', 1577957220, 1773829740, 1127)
('5285.KL', 1714457820, 1773788400, 334)
('5347.KL', 1577919600, 1773961200, 4456)
('5819.KL', 1577957220, 1773961200, 4618)
('6947.KL', 1577957220, 1773788400, 1731)
('8869.KL', 1578275100, 1773926400, 1309)


### 5.3 — Body Fetch Progress

In [7]:
# How many free articles have had their body text collected vs. still remaining?
# CASE WHEN ... THEN 1 ELSE 0 END inside SUM() counts only rows matching a condition.
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
cursor.execute('''
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN body_fetched = 1 THEN 1 ELSE 0 END) AS fetched,
        SUM(CASE WHEN body_fetched = 0 THEN 1 ELSE 0 END) AS remaining
    FROM article_table
    WHERE paywalltype = 'Complimentary'
    AND source = 'thestar'
''')
row = cursor.fetchone()
print(f'Total: {row[0]}, Fetched: {row[1]}, Remaining: {row[2]}')
connection.close()

Total: 27758, Fetched: 27750, Remaining: 8


### 5.4 — Sample Article Preview

In [8]:
# Preview a few rows from the database to check what the data looks like.
# LIMIT 5 means only the first 5 matching rows are returned.
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
cursor.execute('''
    SELECT ticker, title, pubdateunix, kicker
    FROM article_table
    LIMIT 5
''')
for row in cursor.fetchall():
    print(row)
connection.close()

('1155.KL', 'Maybank sells entire interest in Alam Maritim', 1773961200, 'Corporate News')
('1155.KL', 'Maybank disposes of entire 19.16% stake in Alam Maritim', 1773913260, 'Corporate News')
('1155.KL', 'Empire Sushi owner signs IPO underwriting agreement with Maybank IB', 1773388500, 'Markets')
('1155.KL', 'CELEBRATE THE FESTIVE SEASON WITH GREATER REWARDS FROM MAYBANK', 1772838000, 'Banking')
('1155.KL', 'Celebrate the festive season with greater rewards from Maybank', 1772838660, 'Starpicks')


---
## Data Collection Summary

| Item | Detail |
|---|---|
| **Primary source** | The Star (thestar.com.my) via Queryly API |
| **Total articles collected** | ~46,930 |
| **Articles with full body text** | ~27,568 (free / Complimentary only) |
| **Articles with description only** | ~9,362 (paywalled — excluded from sentiment analysis) |
| **Coverage period** | Approximately 2020 to present |
| **Stocks covered** | 10 tickers on Bursa Malaysia |

### Sources Investigated but Excluded

| Source | Reason for Exclusion |
|---|---|
| **The Edge Malaysia** | Search results are rendered client-side via JavaScript and are not present in the raw HTML returned by `requests`. A Playwright-based browser automation approach was investigated as an alternative but was deprioritised due to Jupyter notebook compatibility constraints and project scope. |
| **Bernama** | IP blocked after initial testing; limited historical coverage (~4 months). |
| **Google News** | No article body text available; poor relevance filtering. |